# PV Systematic v4 — Full Pipeline + ZoLa + QA + Auto-Export
Run Stages 1–10.

### Parameters

In [41]:
# Parameters

#original 
# PV_N_LAT=40.73230; PV_S_LAT=40.71700; PV_W_LON=-73.98660; PV_E_LON=-73.97400

#new. i need this for the lines in this cell. im not sure what it does beyond that. 
PV_N_LAT=40.7358876957036; PV_S_LAT=40.71195687391828; PV_W_LON= -74.00188877257987; PV_E_LON= -73.9721415592451




NYC_SODA_APP_TOKEN=''  # optional
IRRADIANCE_PNG=''      # optional PNG path
import os

#what is this part? 
os.environ['PV_N_LAT']=str(PV_N_LAT); os.environ['PV_S_LAT']=str(PV_S_LAT)
os.environ['PV_W_LON']=str(PV_W_LON); os.environ['PV_E_LON']=str(PV_E_LON)
if NYC_SODA_APP_TOKEN: os.environ['NYC_SODA_APP_TOKEN']=NYC_SODA_APP_TOKEN
print('Params set')

Params set


## Stage 1: Fetch Selection (PLUTO)

In [42]:
import importlib
import scripts.fetch_selection as s1
importlib.reload(s1); s1.main()

Wrote all_walkups_6story.csv and all_walkups_6story.geojson


## Stage 2: Fetch Footprints

In [ ]:
# Stage 2 — Fetch Footprints (robust: read via GeoPandas/Fiona)
import os, json, tempfile, requests
import geopandas as gpd
from shapely.geometry import Polygon
import folium


# original bounds 
# N_LAT = float(os.getenv("PV_N_LAT", "40.73230"))
# S_LAT = float(os.getenv("PV_S_LAT", "40.71700"))
# W_LON = float(os.getenv("PV_W_LON", "-73.98660"))
# E_LON = float(os.getenv("PV_E_LON", "-73.97400"))



# new (correct) area of interest encompassing box 
N_LAT = float(os.getenv("PV_N_LAT", "40.7358876957036"))
S_LAT = float(os.getenv("PV_S_LAT", "40.71195687391828"))
W_LON = float(os.getenv("PV_W_LON", "-74.00188877257987"))
E_LON = float(os.getenv("PV_E_LON", "-73.9721415592451"))


 
 #idk what this is 
# NW_CORNER = (40.73474641426536, -73.99075907808152)
# NE_CORNER = (40.72693173607943, -73.9721011685759)
# SE_CORNER=(40.71240175277973, -73.97800218044122)
# SW_CORNER=(40.713096501215226,-74.00733931845855)



FOOTPRINTS_URL = "https://data.cityofnewyork.us/resource/5zhs-2jue.geojson"
APP_TOKEN = os.getenv("NYC_SODA_APP_TOKEN", "").strip()
HEADERS = {"Accept": "application/json"}
if APP_TOKEN:
    HEADERS["X-App-Token"] = APP_TOKEN

params = {
    "$select": "*",
        "$where": f"within_box(the_geom,{S_LAT},{W_LON},{N_LAT},{E_LON})",

    "$limit": 50000,
}






# 1) request once, save exact payload
r = requests.get(FOOTPRINTS_URL, params=params, headers=HEADERS, timeout=180)
try:
    r.raise_for_status()
except Exception as e:
    print("[ERR] HTTP:", e)
    print("[ERR] URL :", r.url)
    print("[ERR] TEXT:", r.text[:600])
    raise

tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".geojson")
tmp_path = tmp.name
tmp.write(r.content)
tmp.close()

# 2) let GeoPandas parse geometries
try:
    gdf = gpd.read_file(tmp_path)
except Exception as e:
    print("[ERR] GeoPandas failed to read the payload:", e)
    print("[HINT] Payload saved to:", tmp_path)
    raise

print("[INFO] Raw features:", len(gdf), "| CRS:", gdf.crs)

# 3) Clean: ensure CRS, fix validity, drop empties and non-(Multi)Polygons
if gdf.crs is None:
    gdf = gdf.set_crs(4326, allow_override=True)

# keep only polygonal
poly_mask = gdf.geom_type.isin(["Polygon","MultiPolygon"])
non_poly = (~poly_mask).sum()
gdf = gdf[poly_mask].copy()

# fix validity with buffer(0)
try:
    gdf["geometry"] = gdf.geometry.buffer(0)
except Exception:
    pass
gdf = gdf[ gdf.geometry.notna() & (~gdf.geometry.is_empty) ].copy()

# area filter: extremely tiny slivers only
dropped_sliver = 0
try:
    a_ft2 = gdf.to_crs(2263).area
    a_m2  = a_ft2 * 0.09290304
    keep  = a_m2 >= float(os.getenv("PV_MIN_POLY_AREA_M2","0.1"))
    dropped_sliver = (~keep).sum()
    gdf = gdf[keep].copy()
except Exception:
    pass

print(f"[INFO] After cleaning: kept={len(gdf)} dropped_nonpoly={non_poly} dropped_sliver={dropped_sliver}")

# 4) Write output or minimal mock if empty
if len(gdf) == 0:
    p1 = Polygon([[W_LON+0.0013, N_LAT-0.0013],[W_LON+0.0016, N_LAT-0.0013],
                  [W_LON+0.0016, N_LAT-0.0016],[W_LON+0.0013, N_LAT-0.0016]])
    p2 = Polygon([[W_LON+0.0018, N_LAT-0.0014],[W_LON+0.0021, N_LAT-0.0014],
                  [W_LON+0.0021, N_LAT-0.0017],[W_LON+0.0018, N_LAT-0.0017]])
    gpd.GeoDataFrame({"mock_id":[1,2]}, geometry=[p1,p2], crs=4326)\
        .to_file("footprints.geojson", driver="GeoJSON")
    print("[WARN] No polygonal footprints after cleaning; wrote 2 mock squares.")
else:
    gdf.to_file("footprints_full.geojson", driver="GeoJSON") # bounding box data
    print("Wrote footprints_full.geojson with", len(gdf), "features")

# Implement polygon 
polygon_coords = [
    (-73.99075, 40.73474), #A: 14th & Broadway 
    ( -73.97209, 40.72689), #B: 14th and FDR 
    (-73.9777,  40.71314), #C: Grand & FDR 
    (-73.98249,  40.71468), #D: Grand and E Broadway  
    (-73.99411, 40.71371), # E : E broadway and Manhattan Bridge  
    (-73.99484, 40.7158), #F: Canal and Manhattan bridge
    (-73.9986, 40.71711),  # G: Canal and Mulberry  
    (-74.00187, 40.71939),  # H: Canal and Broadway  
    (-73.99143, 40.73179),  # I: Broadway and 10th 
     (-73.99075, 40.73474)  #A: 14th & Broadway, close polygon
]

poly = Polygon(polygon_coords)
poly_gdf = gpd.GeoDataFrame(geometry=[poly], crs="EPSG:4326")

footprints_clipped = gpd.clip(gdf, poly_gdf)
footprints_clipped.to_file("footprints.geojson", driver="GeoJSON") # polygon-clipped data (used downstream)

#debugging perimetter border box 
from shapely.geometry import box

bbox_geom = box(W_LON, S_LAT, E_LON, N_LAT)
bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_geom], crs="EPSG:4326")

poly_boundary = poly_gdf.boundary


import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8,8))

# full footprints (light gray for context)
gdf.plot(ax=ax, alpha=0.8)

# clipped footprints (green)
footprints_clipped.plot(ax=ax, alpha=1)

# bounding box (BLUE)
bbox_gdf.boundary.plot(ax=ax, linewidth=3)

# polygon (RED)
poly_gdf.boundary.plot(ax=ax, linewidth=3)

plt.title("AOI Debug: Box (blue) vs Polygon (red)")
plt.show()


[INFO] Raw features: 7384 | CRS: EPSG:4326
[INFO] After cleaning: kept=7384 dropped_nonpoly=0 dropped_sliver=0
Wrote footprints_full.geojson with 7384 features


/var/folders/vq/kf1q0fcx1xsgmxc_k_tqwlg00000gn/T/ipykernel_6731/4083787965.py:161: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [44]:
import geopandas as gpd
gpd.read_file("footprints.geojson").head(2)

,name,base_bbl,shape_area,construction_year,ground_elevation,mappluto_bbl,objectid,feature_code,shape_length,height_roof,last_status_type,bin,last_edited_date,geom_source,doitt_id,geometry
0,None,1002830072,179.4140625,1900,33,1002830072,40252,2100,55.584620815361589,69.14,Constructed,1003598,2017-08-22 18:32:46+00:00,Photogrammetric,480851,"POLYGON ((-73.99045 40.71426, -73.99045 40.714..."
1,None,1002830050,356.57421875,1900,39,1002830050,494626,2100,76.244511923822316,77.24,Constructed,1003583,2017-08-22 17:45:31+00:00,Photogrammetric,595201,"POLYGON ((-73.99222 40.71411, -73.9922 40.7141..."


## Stage 3: PV from Footprints

In [45]:
# Stage 3b — Enrich PV CSV from selection via stable IDs (no reliance on joined column names)
import pandas as pd, geopandas as gpd

# Load your updated building footprints (Stage 2 output)
area_of_interest_footprints = gpd.read_file("footprints.geojson")
PV_POLY = "pv_canopy_footprints.geojson"
PV_CSV  = "pv_footprints_by_building.csv"
SEL_CSV = "all_walkups_6story.csv"

# PLUTO attrs we want copied into PV CSV
ATTACH = ["bbl","address","bldgclass","numfloors","unitsres","yearbuilt"]

# --- 1) Load PV polygons and selection CSV (validate schema) ---
pv = gpd.read_file(PV_POLY)
if pv.crs is None: pv.set_crs(4326, inplace=True)

sel_df = pd.read_csv(SEL_CSV)
required = ATTACH + ["latitude","longitude"]
missing = [c for c in required if c not in sel_df.columns]
if missing:
    raise ValueError(f"{SEL_CSV} missing required columns: {missing}")

# Build selection GeoDataFrame with a stable id
sel_df = sel_df.reset_index(drop=True).copy()
sel_df["sel_id"] = sel_df.index  # stable
sel_gdf = gpd.GeoDataFrame(
    sel_df,
    geometry=gpd.points_from_xy(sel_df["longitude"], sel_df["latitude"]),
    crs=4326
)

# --- 2) Project to State Plane (feet) and build pv_row key for robust merging ---
pv2263  = pv.to_crs(2263).reset_index(drop=False).rename(columns={"index":"pv_row"})
sel2263 = sel_gdf.to_crs(2263).reset_index(drop=True)

# --- 3) Nearest-neighbor join (≤ 40 m; retry 60 m if needed) ---
def do_join(max_ft):
    return gpd.sjoin_nearest(
        sel2263[["sel_id","geometry"]],  # only carry sel_id + geometry into the join
        pv2263[["pv_row","geometry"]],   # only carry pv_row + geometry into the join
        how="left", max_distance=max_ft, distance_col="dist_ft"
    )

hits = do_join(131)   # ~40 m
if "pv_row" not in hits.columns or hits["pv_row"].isna().all():
    print("[WARN] No nearest matches within 40 m — trying 60 m.")
    hits = do_join(197)  # ~60 m

if "pv_row" not in hits.columns or hits["pv_row"].isna().all():
    print("[WARN] Still no matches — PV CSV will be left unchanged.")
    # ensure PV CSV exists with canonical columns
    out_cols = ["bbl","address","bldgclass","numfloors","yearbuilt",
                "roof_area_m2_actual","canopy_area_m2","pv_kw_dc",
                "annual_kwh","avg_daily_kwh","psh_daily_kwh"]
    base = pv.copy()
    for c in out_cols:
        if c not in base.columns: base[c] = None
    base[out_cols].to_csv(PV_CSV, index=False)
else:
    # --- 4) For each pv_row, pick the first sel_id (nearest) ---
    hits = hits.dropna(subset=["pv_row"]).copy()
    hits["pv_row"] = hits["pv_row"].astype(int)
    # first sel_id per pv_row
    map_df = hits.groupby("pv_row")["sel_id"].first().reset_index()

    # Pull desired attributes from the ORIGINAL selection CSV by sel_id
    attrs = map_df.merge(sel_df[["sel_id"] + ATTACH], on="sel_id", how="left")

    # --- 5) Attach to PV polygons (by pv_row), prefer attached values ---
    pv_en = pv.reset_index(drop=False).rename(columns={"index":"pv_row"})
    pv_en = pv_en.merge(attrs.drop(columns=["sel_id"]), on="pv_row", how="left", suffixes=("", "_sel"))

    # Ensure canonical columns exist; copy from *_sel if present
    for c in ATTACH:
        if c not in pv_en.columns:
            pv_en[c] = None
        selcol = c  # after merge, names are canonical unless collision; handle explicit *_sel
        if f"{c}_sel" in pv_en.columns:
            selcol = f"{c}_sel"
        pv_en[c] = pv_en[selcol].combine_first(pv_en[c])
        if f"{c}_sel" in pv_en.columns:
            pv_en.drop(columns=[f"{c}_sel"], inplace=True, errors="ignore")

    # --- 6) Write enriched CSV with canonical names ---
    out_cols = ["bbl","address","bldgclass","numfloors","yearbuilt",
                "roof_area_m2_actual","canopy_area_m2","pv_kw_dc",
                "annual_kwh","avg_daily_kwh","psh_daily_kwh"]
    for c in out_cols:
        if c not in pv_en.columns: pv_en[c] = None
    pv_en[out_cols].to_csv(PV_CSV, index=False)
    print("[OK] Enriched PV CSV written:", PV_CSV)


    # clip canopy geojson with footprints geojson 
    clipped_canopy = gpd.clip(pv, area_of_interest_footprints)
    clipped_canopy.to_file("pv_canopy_footprints_clipped.geojson", driver="GeoJSON")

# --- 7) Report ---
chk = pd.read_csv(PV_CSV)
print("Rows:", len(chk))
print("Non-null PLUTO counts:")
print(chk[["bbl","address","bldgclass","numfloors","yearbuilt"]].notna().sum())
print("\nSample enriched rows:")
print(chk[chk["bbl"].notna()].head(5))

# print(pv.crs)
# print(area_of_interest_footprints.crs)



[OK] Enriched PV CSV written: pv_footprints_by_building.csv
Rows: 344
Non-null PLUTO counts:
bbl          344
address      344
bldgclass    344
numfloors    323
yearbuilt    323
dtype: int64

Sample enriched rows:
            bbl             address bldgclass  numfloors  yearbuilt  \
0  1.002830e+09   162 EAST BROADWAY        C7        6.0     1900.0   
1  1.002820e+09    94 EAST BROADWAY        C7        6.0     1900.0   
2  1.002930e+09  88 DIVISION STREET        C7        6.0     1900.0   
3  1.002920e+09   5 ELDRIDGE STREET        C7        6.0     1900.0   
4  1.002920e+09   7 ELDRIDGE STREET        C7        6.0     1900.0   

   roof_area_m2_actual  canopy_area_m2   pv_kw_dc    annual_kwh  \
0                  NaN       87.523823  17.504765  21670.898598   
1                  NaN      112.166751  22.433350  27772.487557   
2                  NaN      129.317645  25.863529  32019.048808   
3                  NaN      149.300620  29.860124  36966.833540   
4                  NaN  

In [46]:
!pip show pandas


Name: pandas
Version: 2.3.3
Summary: Powerful data structures for data analysis, time series, and statistics
Home-page: https://pandas.pydata.org
Author: 
Author-email: The Pandas Development Team <pandas-dev@python.org>
License: BSD 3-Clause License
         
         Copyright (c) 2008-2011, AQR Capital Management, LLC, Lambda Foundry, Inc. and PyData Development Team
         All rights reserved.
         
         Copyright (c) 2011-2023, Open source contributors.
         
         Redistribution and use in source and binary forms, with or without
         modification, are permitted provided that the following conditions are met:
         
         * Redistributions of source code must retain the above copyright notice, this
           list of conditions and the following disclaimer.
         
         * Redistributions in binary form must reproduce the above copyright notice,
           this list of conditions and the following disclaimer in the documentation
           and/or o

In [47]:
import pandas as pd
pv = pd.read_csv("pv_footprints_by_building.csv")
print(pv[["bbl","address","bldgclass","numfloors","yearbuilt"]].notna().sum())


bbl          344
address      344
bldgclass    344
numfloors    323
yearbuilt    323
dtype: int64


In [48]:
pv[pv["bbl"].notna()].head(10)


,bbl,address,bldgclass,numfloors,yearbuilt,roof_area_m2_actual,canopy_area_m2,pv_kw_dc,annual_kwh,avg_daily_kwh,psh_daily_kwh
0,1.002830e+09,162 EAST BROADWAY,C7,6.0,1900.0,NaN,87.523823,17.504765,21670.898598,59.372325,64.417534
1,1.002820e+09,94 EAST BROADWAY,C7,6.0,1900.0,NaN,112.166751,22.433350,27772.487557,76.089007,82.554729
2,1.002930e+09,88 DIVISION STREET,C7,6.0,1900.0,NaN,129.317645,25.863529,32019.048808,87.723421,95.177786
3,1.002920e+09,5 ELDRIDGE STREET,C7,6.0,1900.0,NaN,149.300620,29.860124,36966.833540,101.278996,109.885256
4,1.002920e+09,7 ELDRIDGE STREET,C7,6.0,1900.0,NaN,108.600186,21.720037,26889.406151,73.669606,79.929737
5,1.002920e+09,9 ELDRIDGE STREET,C7,6.0,1910.0,NaN,105.537512,21.107502,26131.087989,71.592022,77.675609
6,1.002920e+09,11 ELDRIDGE STREET,C7,6.0,1900.0,NaN,156.717286,31.343457,38803.200028,106.310137,115.343923
7,1.002920e+09,13 ELDRIDGE STREET,C7,6.0,1900.0,NaN,189.502374,37.900475,46920.787818,128.550104,139.473747
8,1.002930e+09,3 ALLEN STREET,C7,6.0,1910.0,NaN,133.514321,26.702864,33058.145917,90.570263,98.266540
9,1.002930e+09,70 CANAL STREET,C7,6.0,1910.0,NaN,142.214111,28.442822,35212.213961,96.471819,104.669586


In [49]:
# Stage 3C — Create persistent, enriched centroids (with Address/BBL backfill)
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

PV_POLY = "pv_canopy_footprints_clipped.geojson"
PV_CSV  = "pv_footprints_by_building.csv"
SEL_CSV = "all_walkups_6story.csv"
OUT_GJ  = "pv_canopy_centroids_enriched.geojson"
OUT_CSV = "pv_canopy_centroids_enriched.csv"
SNAP_METERS = 60  # radius for address/BBL backfill

assert os.path.exists(PV_POLY), "Run Stage 3 first (pv_canopy_footprints_clipped.geojson)."

# 1) Load PV polygons, compute centroids from *these* polygons (guarantees alignment)
pv = gpd.read_file(PV_POLY)
if pv.crs is None:
    pv.set_crs(4326, inplace=True)
elif pv.crs.to_epsg() != 4326:
    pv = pv.to_crs(4326)

cent = pv.to_crs(2263).centroid.to_crs(4326)
cent_gdf = gpd.GeoDataFrame(pv.drop(columns=["geometry"]).copy(),
                            geometry=cent, crs="EPSG:4326")

# 2) Merge PV metrics from CSV (index-aligned fast path)
if os.path.exists(PV_CSV):
    dfpv = pd.read_csv(PV_CSV)
    if len(dfpv) == len(cent_gdf):
        for c in ["pv_kw_dc","psh_daily_kwh","annual_kwh",
                  "roof_area_m2_actual","canopy_area_m2",
                  "bbl","address","bldgclass","numfloors","yearbuilt"]:
            if c in dfpv.columns:
                cent_gdf[c] = dfpv[c].values

# 3) Backfill Address/BBL by snapping to selection CSV (≤ 60 m)
if os.path.exists(SEL_CSV):
    sel = pd.read_csv(SEL_CSV)
    if {"bbl","address","latitude","longitude"}.issubset(sel.columns):
        sel_g = gpd.GeoDataFrame(
            sel.copy(),
            geometry=gpd.points_from_xy(sel["longitude"], sel["latitude"]),
            crs="EPSG:4326"
        )
        sel2263 = sel_g.to_crs(2263)[["bbl","address","geometry"]]
        cent2263 = cent_gdf.to_crs(2263).reset_index(drop=False).rename(columns={"index":"pv_row"})
        max_ft = SNAP_METERS * 3.28084

        hits = gpd.sjoin_nearest(
            cent2263[["pv_row","geometry"]],
            sel2263[["bbl","address","geometry"]],
            how="left",
            max_distance=max_ft
        )

        if "pv_row" in hits.columns:
            attach = hits.dropna(subset=["pv_row"])[["pv_row","bbl","address"]]
            cent_gdf = cent_gdf.reset_index(drop=False).rename(columns={"index":"pv_row"}).merge(
                attach, on="pv_row", how="left", suffixes=("","_bf")
            )
            # prefer existing, then backfilled
            if "bbl_bf" in cent_gdf.columns:
                cent_gdf["bbl"] = cent_gdf["bbl"].combine_first(cent_gdf["bbl_bf"])
                cent_gdf.drop(columns=["bbl_bf"], inplace=True, errors="ignore")
            if "address_bf" in cent_gdf.columns:
                cent_gdf["address"] = cent_gdf["address"].combine_first(cent_gdf["address_bf"])
                cent_gdf.drop(columns=["address_bf"], inplace=True, errors="ignore")
            cent_gdf.drop(columns=["pv_row"], inplace=True, errors="ignore")
            cent_gdf.set_crs(4326, inplace=True)

# 4) Write GeoJSON + CSV (with lon/lat)
cent_gdf.to_file(OUT_GJ, driver="GeoJSON")
out = cent_gdf.copy()
out["lon"] = out.geometry.x
out["lat"] = out.geometry.y
out.drop(columns=["geometry"]).to_csv(OUT_CSV, index=False)
print(f"[OK] Wrote {OUT_GJ} and {OUT_CSV}  | rows={len(out)}")


[OK] Wrote pv_canopy_centroids_enriched.geojson and pv_canopy_centroids_enriched.csv  | rows=344


## Stage 4: Maps

In [50]:
# Stage 4 (patched + snapped overlay) — PV overlays with sanitized props + snapped selection→footprint highlight
import os, json, math, datetime as dt
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MiniMap, MeasureControl, Fullscreen, MousePosition, MarkerCluster
import numpy as np

# --- Config ---
SNAP_METERS_PRIMARY = 40      # first try ~40 m
SNAP_METERS_FALLBACK = 60     # fallback to ~60 m if nothing matched

def sanitize_gdf_props(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Return a copy with all non-geometry columns JSON-serializable."""
    if gdf is None or len(gdf) == 0:
        return gdf
    g = gdf.copy()
    for col in g.columns:
        if col == "geometry":
            continue
        s = g[col]
        if pd.api.types.is_datetime64_any_dtype(s):
            g[col] = s.dt.tz_localize(None).dt.strftime("%Y-%m-%d %H:%M:%S")
            continue
        if s.dtype == "object":
            def _clean(v):
                if isinstance(v, pd.Timestamp):
                    return v.tz_localize(None).strftime("%Y-%m-%d %H:%M:%S")
                if isinstance(v, (dt.datetime, dt.date)):
                    return v.strftime("%Y-%m-%d %H:%M:%S") if isinstance(v, dt.datetime) else v.isoformat()
                if isinstance(v, (np.generic,)):
                    try: return v.item()
                    except Exception: return str(v)
                if isinstance(v, (list, dict, set, tuple)):
                    return str(v)
                return v
            g[col] = s.map(_clean)
    return g

def _read(path):
    if not os.path.exists(path): print("[MISS]", path); return None
    try:
        g = gpd.read_file(path)
        if g is None or len(g)==0:
            print("[EMPTY]", path); return None
        return g
    except Exception as e:
        print("[ERR]", path, e); return None

# Load layers
fp  = _read("footprints.geojson")
pv  = _read("pv_canopy_footprints.geojson")
cen = _read("pv_canopy_centroids.geojson")
sel_gj = _read("all_walkups_6story.geojson")

# If selection GeoJSON is missing, try from CSV (latitude/longitude)
if sel_gj is None and os.path.exists("all_walkups_6story.csv"):
    sdf = pd.read_csv("all_walkups_6story.csv")
    lower = {c.lower(): c for c in sdf.columns}
    if "latitude" in lower and "longitude" in lower:
        sel_gj = gpd.GeoDataFrame(
            sdf.copy(),
            geometry=gpd.points_from_xy(sdf[lower["longitude"]], sdf[lower["latitude"]]),
            crs="EPSG:4326"
        )
        print("[INFO] Built selection points from CSV.")

# Normalize CRS to EPSG:4326
for name, g in [("Footprints", fp), ("PV footprints", pv), ("PV centroids", cen), ("Selection", sel_gj)]:
    if g is not None:
        if g.crs is None:
            g.set_crs(4326, inplace=True)
        elif g.crs.to_epsg() != 4326:
            g.to_crs(4326, inplace=True)
        print(f"[INFO] {name}: {len(g)} features")

# --- Create snapped selection→footprint overlay ---
selected_fp = None
if fp is not None and sel_gj is not None and len(fp)>0 and len(sel_gj)>0:
    try:
        # Work in State Plane (feet) for distance
        fp2263  = fp.to_crs(2263).reset_index(drop=False).rename(columns={"index":"pv_row"})
        sel2263 = sel_gj.to_crs(2263).reset_index(drop=True)
        def do_join(meters):
            max_ft = meters * 3.28084
            return gpd.sjoin_nearest(
                sel2263[["geometry"]], fp2263[["pv_row","geometry"]],
                how="left", max_distance=max_ft
            )
        hits = do_join(SNAP_METERS_PRIMARY)
        if "pv_row" not in hits.columns or hits["pv_row"].isna().all():
            print(f"[WARN] No nearest matches within {SNAP_METERS_PRIMARY} m — trying {SNAP_METERS_FALLBACK} m.")
            hits = do_join(SNAP_METERS_FALLBACK)
        if "pv_row" in hits.columns and hits["pv_row"].notna().any():
            pv_rows = sorted(set(hits["pv_row"].dropna().astype(int).tolist()))
            selected_fp = fp2263.loc[pv_rows].to_crs(4326)
            print(f"[INFO] Selected footprints (snapped): {len(selected_fp)}")
        else:
            print("[INFO] No selection matched footprints for highlight.")
    except Exception as e:
        print("[WARN] Snapped selection overlay failed:", e)

# Sanitize properties (avoid Timestamp errors)
fp  = sanitize_gdf_props(fp)  if fp  is not None else None
pv  = sanitize_gdf_props(pv)  if pv  is not None else None
cen = sanitize_gdf_props(cen) if cen is not None else None
sel_gj = sanitize_gdf_props(sel_gj) if sel_gj is not None else None
selected_fp = sanitize_gdf_props(selected_fp) if selected_fp is not None else None

# Compute bounds and center
cands = [g for g in [fp,pv,cen,sel_gj,selected_fp] if g is not None and len(g)>0]
if not cands:
    raise RuntimeError("No layers to map. Generate earlier stages first.")
minx=miny= 1e9; maxx=maxy=-1e9
for g in cands:
    mnx,mny,mxx,mxy = g.total_bounds
    minx,miny,maxx,maxy = min(minx,mnx),min(miny,mny),max(maxx,mxx),max(maxy,mxy)
ctr_lat=(miny+maxy)/2; ctr_lon=(minx+maxx)/2

# Build map
m = folium.Map(location=[ctr_lat,ctr_lon], zoom_start=15, tiles="OpenStreetMap", control_scale=True)
Fullscreen().add_to(m)
MiniMap(toggle_display=True, position="bottomright").add_to(m)
MeasureControl(primary_length_unit="meters").add_to(m)
MousePosition(prefix="Lat/Lon", position="bottomleft").add_to(m)

def add_geojson(g, name, fill, stroke, weight=1, fill_opacity=0.3, show=True, tooltip_fields=None):
    gj = folium.GeoJson(
        data=json.loads(g.to_json()),
        name=name,
        show=show,
        style_function=lambda _:{ "fillColor":fill, "color":stroke, "weight":weight, "fillOpacity":fill_opacity }
    )
    if tooltip_fields:
        fields = [c for c in tooltip_fields if c in g.columns and c!="geometry"][:12]
        if fields:
            gj.add_child(folium.features.GeoJsonTooltip(fields=fields))
    gj.add_to(m)

# Layers
if fp is not None and len(fp)>0:
    add_geojson(fp, "Footprints (all)", fill="#9ecae1", stroke="#6baed6", weight=1, fill_opacity=0.15, show=False,
                tooltip_fields=list(fp.columns))

if pv is not None and len(pv)>0:
    add_geojson(pv, "PV Canopy Footprints", fill="#ffb347", stroke="#f59f00", weight=1, fill_opacity=0.35, show=True,
                tooltip_fields=list(pv.columns))

if selected_fp is not None and len(selected_fp)>0:
    add_geojson(selected_fp, "Selected Footprints (snapped)", fill="#3182bd", stroke="#08519c", weight=3, fill_opacity=0.25, show=True,
                tooltip_fields=list(selected_fp.columns))

# Centroid markers
if cen is not None and len(cen)>0:
    grp = folium.FeatureGroup(name="PV Centroids", show=False)
    for _,row in cen.iterrows():
        try:
            lat = row.geometry.y; lon = row.geometry.x
            kv = []
            for k in ["pv_kw_dc","annual_kwh","roof_area_m2_actual","canopy_area_m2","bbl","address","bldgclass","numfloors","yearbuilt"]:
                if k in cen.columns and pd.notna(row.get(k, None)):
                    kv.append(f"<b>{k}</b>: {row[k]}")
            html = "<br>".join(kv) if kv else "PV centroid"
            folium.CircleMarker([lat,lon], radius=5, color="#7b1fa2", fill=True, fill_opacity=0.8,
                                popup=folium.Popup(html, max_width=360)).add_to(grp)
        except Exception:
            continue
    grp.add_to(m)

# Selection points (cluster)
if sel_gj is not None and len(sel_gj)>0 and sel_gj.geom_type.isin(["Point"]).any():
    cl = MarkerCluster(name="Selection Points", show=False)
    for _,r in sel_gj.iterrows():
        try:
            lat=r.geometry.y; lon=r.geometry.x
            lab = r.get("address") or r.get("bbl") or "Selected"
            folium.Marker([lat,lon], tooltip=str(lab)).add_to(cl)
        except Exception:
            continue
    cl.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[miny,minx],[maxy,maxx]])

os.makedirs("maps", exist_ok=True)
m.save("maps/pv_overlays_map.html")
print("Saved maps/pv_overlays_map.html")

# Optional: PV-only zoomed map

if pv is not None and len(pv)>0:
    mnx,mny,mxx,mxy = pv.total_bounds
    m2 = folium.Map(location=[(mny+mxy)/2,(mnx+mxx)/2], zoom_start=16, tiles="OpenStreetMap")
    add_geojson(pv, "PV Canopy Footprints", fill="#ffb347", stroke="#f59f00", weight=1, fill_opacity=0.40, show=True,
                tooltip_fields=list(pv.columns))
    if selected_fp is not None and len(selected_fp)>0:
        add_geojson(selected_fp, "Selected Footprints (snapped)", fill="#3182bd", stroke="#08519c", weight=3, fill_opacity=0.25, show=True,
                    tooltip_fields=list(selected_fp.columns))
    folium.LayerControl().add_to(m2)
    m2.fit_bounds([[mny,mnx],[mxy,mxx]])
    m2.save("maps/pv_overlays_map_pvonly.html")
    print("Saved maps/pv_overlays_map_pvonly.html")




[INFO] Footprints: 4863 features
[INFO] PV footprints: 344 features
[INFO] PV centroids: 344 features
[INFO] Selection: 809 features
[WARN] No nearest matches within 40 m — trying 60 m.
[INFO] No selection matched footprints for highlight.
Saved maps/pv_overlays_map.html
Saved maps/pv_overlays_map_pvonly.html


In [51]:
# Stage 4C — PV-only map with sanitized properties
import os, json
import geopandas as gpd
import folium
import pandas as pd
import numpy as np

def sanitize_gdf_props(g):
    if g is None or len(g)==0: return g
    g = g.copy()
    for c in g.columns:
        if c=="geometry": continue
        s = g[c]
        if pd.api.types.is_datetime64_any_dtype(s):
            g[c] = s.dt.tz_localize(None).dt.strftime("%Y-%m-%d %H:%M:%S")
        elif s.dtype=="object":
            g[c] = s.map(lambda v: v.isoformat() if hasattr(v,"isoformat") else (v.item() if hasattr(v,"item") else (str(v) if isinstance(v,(list,dict,set,tuple)) else v)))
    return g

pv_path = "pv_canopy_footprints.geojson"
pv = gpd.read_file(pv_path) if os.path.exists(pv_path) else None
if pv is None or len(pv)==0: raise RuntimeError("No PV polygons found. Run Stage 3 first.")
if pv.crs is None: pv.set_crs(4326, inplace=True)
elif pv.crs.to_epsg()!=4326: pv.to_crs(4326, inplace=True)
pv = sanitize_gdf_props(pv)

mnx,mny,mxx,mxy = pv.total_bounds
m = folium.Map(location=[(mny+mxy)/2,(mnx+mxx)/2], zoom_start=16, tiles="OpenStreetMap", control_scale=True)

def add_geojson(g, name, fill, stroke, weight=1, fill_opacity=0.4, show=True, tooltip_fields=None):
    gj = folium.GeoJson(
        data=json.loads(g.to_json()),
        name=name,
        show=show,
        style_function=lambda _:{ "fillColor":fill, "color":stroke, "weight":weight, "fillOpacity":fill_opacity }
    )
    if tooltip_fields:
        fields=[c for c in tooltip_fields if c in g.columns and c!="geometry"][:12]
        if fields: gj.add_child(folium.features.GeoJsonTooltip(fields=fields))
    gj.add_to(m)

add_geojson(pv, "PV Canopy Footprints", "#ffb347", "#f59f00", 1, 0.40, True, list(pv.columns))
folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[mny,mnx],[mxy,mxx]])

os.makedirs("maps", exist_ok=True)
m.save("maps/pv_overlays_map_pvonly.html")
print("Saved maps/pv_overlays_map_pvonly.html")


Saved maps/pv_overlays_map_pvonly.html


In [52]:
# Stage 4D+ — Consolidated irradiance map with Stage-8 overlays + enriched centroids + legend

import os, io, base64, json, datetime as dt
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------- Inputs ----------
FOOTPRINTS_PATH = "footprints.geojson"                           # polygon-clipped data (used downstream)
PV_POLY_PATH    = "pv_canopy_footprints.geojson"                 # optional but recommended
CENT_ENRICH_GJ  = "pv_canopy_centroids_enriched.geojson"         # produced by Stage 3C (required for addresses/BBL)
SEL_POINTS_GJ   = "all_walkups_6story.geojson"                   # optional (selection points)
OUT_HTML        = "maps/irradiance_timeseries_map_fast.html"     # output file

# ---------- Helpers ----------
def sanitize_gdf_props(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Return copy with JSON-serializable non-geometry columns."""
    if gdf is None or len(gdf) == 0:
        return gdf
    g = gdf.copy()
    for col in g.columns:
        if col == "geometry":
            continue
        s = g[col]
        if pd.api.types.is_datetime64_any_dtype(s):
            g[col] = s.dt.tz_localize(None).dt.strftime("%Y-%m-%d %H:%M:%S")
            continue
        if s.dtype == "object":
            def _clean(v):
                if isinstance(v, pd.Timestamp):
                    return v.tz_localize(None).strftime("%Y-%m-%d %H:%M:%S")
                if isinstance(v, (dt.datetime, dt.date)):
                    return v.strftime("%Y-%m-%d %H:%M:%S") if isinstance(v, dt.datetime) else v.isoformat()
                if isinstance(v, np.generic):
                    try: return v.item()
                    except Exception: return str(v)
                if isinstance(v, (list, dict, set, tuple)):
                    return str(v)
                return v
            g[col] = s.map(_clean)
    return g

def prep_for_map(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Ensure EPSG:4326 and JSON-safe props."""
    if gdf is None or len(gdf) == 0:
        return gdf
    if gdf.crs is None: gdf = gdf.set_crs(4326)
    elif gdf.crs.to_epsg() != 4326: gdf = gdf.to_crs(4326)
    return sanitize_gdf_props(gdf)

def _read(path):
    if not os.path.exists(path):
        print("[MISS]", path); return None
    try:
        g = gpd.read_file(path)
        if g is None or len(g) == 0:
            print("[EMPTY]", path); return None
        return g
    except Exception as e:
        print("[ERR]", path, e); return None

# ---------- Load layers ----------
assert os.path.exists(CENT_ENRICH_GJ), "Run Stage 3C first (creates pv_canopy_centroids_enriched.geojson)."
cent = prep_for_map(_read(CENT_ENRICH_GJ))
fp   = prep_for_map(_read(FOOTPRINTS_PATH))
pv   = prep_for_map(_read(PV_POLY_PATH))
sel  = prep_for_map(_read(SEL_POINTS_GJ))

# ---------- Aggregate metrics for legend ----------
pv_kw = pd.to_numeric(cent.get("pv_kw_dc"), errors="coerce")
psh   = pd.to_numeric(cent.get("psh_daily_kwh"), errors="coerce")

total_kw = float(np.nansum(pv_kw)) if pv_kw is not None else 0.0
if "annual_kwh" in cent.columns and pd.notna(cent["annual_kwh"]).any():
    total_annual_kwh = float(np.nansum(pd.to_numeric(cent["annual_kwh"], errors="coerce")))
else:
    psh_fill = psh.fillna(3.8) if psh is not None else pd.Series([3.8]*len(cent))
    total_annual_kwh = float(np.nansum((pv_kw.fillna(0) * psh_fill) * 365.0))

total_mw  = total_kw / 1_000.0
total_gw  = total_kw / 1_000_000.0
total_mwh = total_annual_kwh / 1_000.0
total_gwh = total_annual_kwh / 1_000_000.0

# Aggregate 14-day (synthetic, deterministic) for legend
rng = np.random.default_rng(42)
days = np.arange(14)
agg_kw_psh = float(np.nansum(pv_kw.fillna(0) * (psh.fillna(3.8) if psh is not None else 3.8)))
agg_daily = agg_kw_psh * (1.0 + 0.08*np.sin(2*np.pi*days/7.0)) * rng.normal(1.0, 0.03, size=14)
agg_daily = np.clip(agg_daily, 0, None)

# Inline BAR chart (base64) for legend
os.makedirs("maps", exist_ok=True)
fig, ax = plt.subplots(figsize=(3.8,1.6), dpi=150)
ax.bar(days+1, agg_daily, width=0.8)
ax.set_title("Aggregate kWh/day (last 14)", fontsize=9)
ax.set_xlabel("Day", fontsize=8); ax.set_ylabel("kWh", fontsize=8)
ax.grid(True, axis="y", linewidth=0.3, alpha=0.5)
plt.tight_layout()
buf = io.BytesIO(); fig.savefig(buf, format="png"); plt.close(fig)
chart_b64 = base64.b64encode(buf.getvalue()).decode("ascii")

# ---------- Build map ----------
# Bounds from whatever layers exist
layers = [g for g in [fp, pv, cent, sel] if g is not None and len(g)>0]
minx=miny= 1e9; maxx=maxy=-1e9
for g in layers:
    mnx,mny,mxx,mxy = g.total_bounds
    minx,miny,maxx,maxy = min(minx,mnx),min(miny,mny),max(maxx,mxx),max(maxy,mxy)
ctr_lat=(miny+maxy)/2; ctr_lon=(minx+maxx)/2

m = folium.Map(location=[ctr_lat, ctr_lon], zoom_start=14, tiles="OpenStreetMap", control_scale=True)

def add_geojson(g, name, fill, stroke, weight=1, fill_opacity=0.3, show=True, tooltip_fields=None):
    gj = folium.GeoJson(
        data=json.loads(g.to_json()),
        name=name,
        show=show,
        style_function=lambda _:{ "fillColor":fill, "color":stroke, "weight":weight, "fillOpacity":fill_opacity }
    )
    if tooltip_fields:
        fields=[c for c in tooltip_fields if c in g.columns and c!="geometry"][:12]
        if fields: gj.add_child(folium.features.GeoJsonTooltip(fields=fields))
    gj.add_to(m)

# Stage-8 overlay parity
if fp is not None and len(fp)>0:
    add_geojson(fp, "Building Footprints", "#9ecae1", "#6baed6", 1, 0.20, show=False, tooltip_fields=list(fp.columns))
if pv is not None and len(pv)>0:
    add_geojson(pv, "PV Canopy Footprints", "#ffb347", "#f59f00", 1, 0.35, show=True, tooltip_fields=list(pv.columns))
if sel is not None and len(sel)>0 and sel.geom_type.isin(["Point"]).any():
    add_geojson(sel, "Selection Points", "#31a354", "#006d2c", 1, 0.25, show=False, tooltip_fields=list(sel.columns))

# Enriched centroids → markers (fast, aligned)
cluster = MarkerCluster(name="PV Sites (enriched centroids)").add_to(m)
for r in cent.itertuples(index=False):
    lat, lon = r.geometry.y, r.geometry.x
    addr = getattr(r,"address",None) or "(address unavailable)"
    bbl  = getattr(r,"bbl",None) or "(BBL unavailable)"
    pvkw = getattr(r,"pv_kw_dc",None)
    pshs = getattr(r,"psh_daily_kwh",None)
    pvkw_txt = f"{float(pvkw):.1f} kW" if pvkw is not None and pd.notna(pvkw) else "n/a"
    psh_txt  = f"{float(pshs):.2f} PSH" if pshs is not None and pd.notna(pshs) else "n/a"
    html = (
        f"<b>Address:</b> {addr}<br/>"
        f"<b>BBL:</b> {bbl}<br/>"
        f"<b>PV size:</b> {pvkw_txt} | <b>PSH:</b> {psh_txt}"
    )
    folium.Marker([lat,lon], tooltip=addr if addr else "PV site",
                  popup=folium.Popup(html, max_width=320)).add_to(cluster)

folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[miny,minx],[maxy,maxx]])

# Fixed legend (kW/MW/GW; kWh/MWh/GWh + bar chart)
legend_html = f"""
<div style="
  position: fixed; bottom: 12px; left: 12px; z-index: 9999;
  width: 390px; background: rgba(255,255,255,0.96);
  border: 1px solid #999; border-radius: 8px; padding: 10px; font-size: 12px;">
  <div style="font-weight:700; margin-bottom:6px;">Aggregate PV Summary</div>
  <div><b>Total DC size:</b> {total_kw:,.1f} kW &nbsp;|&nbsp; {total_mw:,.3f} MW &nbsp;|&nbsp; {total_gw:,.6f} GW</div>
  <div><b>Total annual energy:</b> {total_annual_kwh:,.0f} kWh/yr &nbsp;|&nbsp; {total_mwh:,.1f} MWh/yr &nbsp;|&nbsp; {total_gwh:,.3f} GWh/yr</div>
  <div style="margin-top:6px;">
    <img src="data:image/png;base64,{chart_b64}" width="360"/>
  </div>
 
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Save
os.makedirs("maps", exist_ok=True)
m.save(OUT_HTML)
print("Saved", OUT_HTML)


Saved maps/irradiance_timeseries_map_fast.html


## Stage 5: Summaries

In [53]:
import scripts.summarize_totals as s; s.main()

Wrote totals_summary.md and totals_bar.png


## Stage 6: ZoLa overlays

In [54]:
import scripts.fetch_zola_layers as s; s.main()

wrote DATA/zola/zoning_districts.geojson features 58
wrote DATA/zola/commercial_overlays.geojson features 124
wrote DATA/zola/special_purpose.geojson features 1
wrote DATA/zola/limited_height.geojson features 0
wrote DATA/zola/mih.geojson features 0
wrote DATA/zola/edesignations.geojson features 0
wrote DATA/zola/effective_firm_2007.geojson features 0
done.


## Stage 7: Historic Districts (optional)

In [55]:
import scripts.fetch_historic_districts as s; s.main()

wrote DATA/zola/historic_districts.geojson features 4


## Stage 8: Enrich with ZoLa

In [56]:
import scripts.enrich_with_zola as s; s.main()

wrote DATA/processed/footprints_enriched_zola.geojson
wrote DATA/processed/footprints_enriched_zola.csv


In [57]:
# --- Stage 8 (drop-in) — sanitized layers + safe centroids fallback + Folium map ---

import os, json, datetime as dt
import numpy as np
import pandas as pd
import geopandas as gpd
import folium

# ---------- Helpers ----------
def sanitize_gdf_props(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Return a copy with all non-geometry columns JSON-serializable:
      - pandas.Timestamp / datetime / date -> ISO strings
      - numpy scalars -> Python scalars
      - lists/dicts/sets/tuples -> str(...)
    """
    if gdf is None or len(gdf) == 0:
        return gdf
    g = gdf.copy()
    for col in g.columns:
        if col == "geometry":
            continue
        s = g[col]
        # bulk-convert datetime-like dtypes
        if pd.api.types.is_datetime64_any_dtype(s):
            g[col] = s.dt.tz_localize(None).dt.strftime("%Y-%m-%d %H:%M:%S")
            continue
        # per-value cleanup for object dtype
        if s.dtype == "object":
            def _clean(v):
                if isinstance(v, pd.Timestamp):
                    return v.tz_localize(None).strftime("%Y-%m-%d %H:%M:%S")
                if isinstance(v, (dt.datetime, dt.date)):
                    return v.strftime("%Y-%m-%d %H:%M:%S") if isinstance(v, dt.datetime) else v.isoformat()
                if isinstance(v, np.generic):
                    try:
                        return v.item()
                    except Exception:
                        return str(v)
                if isinstance(v, (list, dict, set, tuple)):
                    return str(v)
                return v
            g[col] = s.map(_clean)
    return g

def prep_for_map(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Ensure EPSG:4326 and JSON-safe properties for Folium."""
    if gdf is None or len(gdf) == 0:
        return gdf
    if gdf.crs is None:
        gdf = gdf.set_crs(4326)
    elif gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(4326)
    return sanitize_gdf_props(gdf)

def _read(path):
    """Read a GeoJSON (or vector) file; return None on missing/empty/error."""
    if not os.path.exists(path):
        print("[MISS]", path); return None
    try:
        g = gpd.read_file(path)
        if g is None or len(g) == 0:
            print("[EMPTY]", path); return None
        return g
    except Exception as e:
        print("[ERR]", path, e); return None

# ---------- Inputs (adjust paths if your notebook uses different ones) ----------
FOOTPRINTS_PATH = "footprints.geojson"
PV_POLY_PATH    = "pv_canopy_footprints.geojson"
PV_CENT_PATH    = "pv_canopy_centroids.geojson"             # optional (raw)
PV_CENT_ENRICH  = "pv_canopy_centroids_enriched.geojson"    # optional (preferred)
SEL_POINTS_GJ   = "all_walkups_6story.geojson"              # optional
OUT_HTML        = "maps/stage8_map.html"

# ---------- Load layers (with safe fallback for centroids) ----------
footprints = _read(FOOTPRINTS_PATH)
pv_poly    = _read(PV_POLY_PATH)

pv_cent = _read(PV_CENT_ENRICH)
if pv_cent is None or len(pv_cent) == 0:
    pv_cent = _read(PV_CENT_PATH)

sel_pts = _read(SEL_POINTS_GJ)

# ---------- Prep layers for Folium ----------
footprints = prep_for_map(footprints)
pv_poly    = prep_for_map(pv_poly)
pv_cent    = prep_for_map(pv_cent)
sel_pts    = prep_for_map(sel_pts)

# ---------- Compute map bounds ----------
layers = [g for g in [footprints, pv_poly, pv_cent, sel_pts] if g is not None and len(g) > 0]
if not layers:
    raise RuntimeError("Stage 8: No layers found. Verify earlier stages wrote their GeoJSON files.")

minx = miny =  1e9
maxx = maxy = -1e9
for g in layers:
    mnx, mny, mxx, mxy = g.total_bounds
    minx, miny, maxx, maxy = min(minx, mnx), min(miny, mny), max(maxx, mxx), max(maxy, mxy)
ctr_lat = (miny + maxy) / 2
ctr_lon = (minx + maxx) / 2

# ---------- Build the map ----------
m = folium.Map(location=[ctr_lat, ctr_lon], zoom_start=15, tiles="OpenStreetMap", control_scale=True)

def add_geojson(g, name, fill, stroke, weight=1, fill_opacity=0.3, show=True, tooltip_fields=None):
    gj = folium.GeoJson(
        data=json.loads(g.to_json()),
        name=name,
        show=show,
        style_function=lambda _:{ "fillColor": fill, "color": stroke, "weight": weight, "fillOpacity": fill_opacity }
    )
    if tooltip_fields:
        fields = [c for c in tooltip_fields if c in g.columns and c != "geometry"][:12]
        if fields:
            gj.add_child(folium.features.GeoJsonTooltip(fields=fields))
    gj.add_to(m)

# Add base layers
if footprints is not None and len(footprints) > 0:
    add_geojson(
        footprints,
        "Building Footprints",
        fill="#9ecae1", stroke="#6baed6", weight=1, fill_opacity=0.20, show=False,
        tooltip_fields=list(footprints.columns)
    )

if pv_poly is not None and len(pv_poly) > 0:
    add_geojson(
        pv_poly,
        "PV Canopy Footprints",
        fill="#ffb347", stroke="#f59f00", weight=1, fill_opacity=0.35, show=True,
        tooltip_fields=list(pv_poly.columns)
    )

# Centroids (if points)
if pv_cent is not None and len(pv_cent) > 0 and pv_cent.geom_type.isin(["Point"]).any():
    grp = folium.FeatureGroup(name="PV Centroids", show=False)
    for _, row in pv_cent.iterrows():
        try:
            lat, lon = row.geometry.y, row.geometry.x
            lab = row.get("address") or row.get("bbl") or "PV centroid"
            folium.CircleMarker(
                [lat, lon],
                radius=4, color="#7b1fa2", fill=True, fill_opacity=0.9,
                tooltip=str(lab)
            ).add_to(grp)
        except Exception:
            continue
    grp.add_to(m)

# Selection points (if present and points)
if sel_pts is not None and len(sel_pts) > 0 and sel_pts.geom_type.isin(["Point"]).any():
    add_geojson(
        sel_pts,
        "Selection Points",
        fill="#31a354", stroke="#006d2c", weight=1, fill_opacity=0.25, show=False,
        tooltip_fields=list(sel_pts.columns)
    )

folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[miny, minx], [maxy, maxx]])

os.makedirs("maps", exist_ok=True)
m.save(OUT_HTML)
print(f"Saved {OUT_HTML}")


Saved maps/stage8_map.html


In [58]:
import importlib
import scripts.make_irradiance_map as irr
importlib.reload(irr); irr.main()

Saved: maps/irradiance_timeseries_map.html


## Stage 9: QA Validation

In [59]:
import subprocess, sys; subprocess.run([sys.executable,'tools/qa_checklist.py'], check=False)

== Stage 1 ==
 all_walkups_6story.csv: OK
 all_walkups_6story.geojson: OK
== Stage 2 ==
 footprints.geojson: OK
== Stage 3 ==
 pv_footprints_by_building.csv: OK
 pv_canopy_footprints.geojson: OK
 pv_canopy_centroids.geojson: OK
 pv_canopy_centroids.csv: OK
== Stage 4 ==
 maps/pv_canopies_map.html: MISSING
== Stage 5 ==
 totals_summary.md: OK
 totals_bar.png: OK
== Stage 6 ==
 DATA/zola/zoning_districts.geojson: OK
== Stage 8 ==
 DATA/processed/footprints_enriched_zola.geojson: OK
 DATA/processed/footprints_enriched_zola.csv: OK
Some outputs missing. Re-run relevant stage(s).


CompletedProcess(args=['/Users/arfachowdhary/miniconda3/envs/solar-map/bin/python', 'tools/qa_checklist.py'], returncode=1)

## Stage 10: Timestamped Export

In [60]:
import time, os, shutil, glob
stamp=time.strftime('%Y%m%d_%H%M%S')
exp=os.path.join('exports', stamp)
os.makedirs(exp, exist_ok=True)
for pat in ['*.csv','*.geojson','*.md','*.png']:
    for f in glob.glob(pat):
        try: shutil.copy2(f, exp)
        except Exception: pass
if os.path.isdir('maps'):
    shutil.copytree('maps', os.path.join(exp,'maps'), dirs_exist_ok=True)
open(os.path.join(exp,'EXPORT_OK.txt'),'w').write('Export completed at '+stamp)
print('Exported artifacts to', exp)

Exported artifacts to exports/20260403_145338
